<!-- nav -->
[← 4 · Values with widths](04_Values_With_Widths.ipynb) · [Home](../../README.md) · [6 · Analyzing and rewriting rules →](06_Analyzing_And_Rewriting_Rules.ipynb)

# 5 · Deciding what you can now

**The problem.** Two questions keep coming back. A new product arrives with its own limits (a frozen vaccine at −25 to
−15 °C, for up to 120 hours) and the rule must be written again, slightly differently. And while a shipment is still on
the road, "will it arrive in time?" depends on a number nobody has yet: how many more hours the trip takes.

**Partial evaluation** answers both. Give it the variables you know, and it evaluates everything they decide, leaving
the rest as an expression: the **residual**, the question that's left.

In [1]:
import { Evaluators, Expressions as E, Partials } from "@mbse/expressions";
import { Symbolics } from "@mbse/expressions/Framework";
import { cold_chain, show } from "./toolkit.ts";

const store = cold_chain();
const [self, low, high, max_hours, to_go] = ["this", "low", "high", "max_hours", "to_go"].map((n) => E.variable(n));
const celsius = self.celsius;
const template = low.le(celsius).and_(celsius.le(high)).and_(self.hours.add(to_go).le(max_hours));
console.log(show(template));
console.log([...Symbolics.free(template.data)].sort());

and(and(le(low, get(this, 'celsius')), le(get(this, 'celsius'), high)), le(add(get(this, 'hours'), to_go), max_hours))


[ "high", "low", "max_hours", "this", "to_go" ]


`Symbolics.free` gives the names a rule needs from its scope: the shipment, the product's limits, and the hours to go.

## Step 1: one template, many products

Binding the limits gives each product its rule:

In [2]:
const vaccine = Partials.OfAny(template, { low: 2.0, high: 8.0, max_hours: 72n });
const frozen = Partials.OfAny(template, { low: -25.0, high: -15.0, max_hours: 120n });
for (const rule of [vaccine, frozen]) console.log(show(rule as any), [...Symbolics.free(rule)].sort());

and(and(le(2.0, get(this, 'celsius')), le(get(this, 'celsius'), 8.0)), le(add(get(this, 'hours'), to_go), 72)) [ "this", "to_go" ]


and(and(le(-25.0, get(this, 'celsius')), le(get(this, 'celsius'), -15.0)), le(add(get(this, 'hours'), to_go), 120)) [ "this", "to_go" ]


The residuals are ordinary expressions: they save, translate and evaluate like any other, and don't refer back to
the template.

## Step 2: what the shipment decides

Now bind a shipment, but not the hours to go. Whatever the shipment decides is evaluated, and Kleene's logic (case
study 2) simplifies what's left: `and(true, x)` is `x`, and `and(false, x)` is false whatever `x` is.

In [3]:
const s1 = store.Shipment().id("S1").celsius(5.5).hours(30n).create();
const s2 = store.Shipment().id("S2").celsius(9.1).hours(30n).create();
const s3 = store.Shipment().id("S3").hours(20n).create(); // no temperature yet
for (const shipment of [s1, s2, s3]) console.log(shipment.id, show(Partials.OfAny(vaccine as any, { this: shipment }) as any));

S1 le(add(30, to_go), 72)


S2 false


S3 and(and(le(2.0, get(this, 'celsius')), le(get(this, 'celsius'), 8.0)), le(add(20, to_go), 72))


- S1 is in range, so all that's left is the clock: it arrives in time if 30 more hours, plus what's to go, stay
  within 72.
- S2 is out of range, so it fails whatever the trip takes.
- S3 has no temperature, so its range check stays as it was, still about `this`, and the clock check is reduced.

## Step 3: the residual agrees with the rule

A residual is only useful if it means the same thing. Finishing it with the rest of the variables gives what the
original gives with all of them:

In [4]:
for (const shipment of [s1, s2, s3]) {
  const residual = Partials.OfAny(vaccine as any, { this: shipment });
  for (const hours of [10n, 50n]) {
    const whole = Evaluators.OfAny(vaccine as any, { this: shipment, to_go: hours });
    if (Evaluators.OfAny(residual as any, { this: shipment, to_go: hours }) !== whole) throw new Error("disagrees");
    console.log(shipment.id, String(hours), whole);
  }
}

S1 10 true


S1 50 false


S2 10 false


S2 50 false


S3 10 null


S3 50 null


## Step 4: lets of known values

A let whose value is known binds it in its body, and disappears once nothing refers to it:

In [5]:
const total = E.let_("hours", self.hours.add(to_go), E.variable("hours").le(72n));
console.log(show(total));
console.log(show(Partials.OfAny(total, { this: s1 }) as any));
console.log(show(Partials.OfAny(total, { this: s1, to_go: 50n }) as any));

let hours = add(get(this, 'hours'), to_go) in le(hours, 72)


let hours = add(30, to_go) in le(hours, 72)


false


With `to_go` unknown, the let stays, now binding `add(30, to_go)`. With both known, the whole rule is a value.

## Why it works this way

- **The residual is the question that's left.** Hand it to whoever knows the rest, such as the routing service that
  knows the hours to go, and they can answer it without knowing anything about shipments.
- **Specializing is evaluating.** A template's parameters are variables like any other, so specializing it needs no
  new mechanism, and the specialized rule is checked and stored like any rule.
- **Closed parts are evaluated whole.** A subexpression whose variables are all known is evaluated, and replaced by its
  value when it has one. When its value is unknown, as S3's temperature is, it stays as written: the residual still
  says what to ask.

<!-- nav -->
[← 4 · Values with widths](04_Values_With_Widths.ipynb) · [Home](../../README.md) · [6 · Analyzing and rewriting rules →](06_Analyzing_And_Rewriting_Rules.ipynb)